# SlowFast R50 — Emotion/Violence Recognition from RGB Video

## Overview
This notebook fine-tunes **SlowFast R50** (`pytorchvideo.models.hub.slowfast_r50`) on RGB video clips.

> **SlowFast** uses two pathways:
> - **Slow pathway** — low frame rate (8 frames), captures spatial semantics
> - **Fast pathway** — high frame rate (32 frames), captures temporal motion
>
> Both pathways are derived from the same 32-frame clip. The slow pathway subsamples every 4th frame (`video[:, :, ::4, :, :]`), so no separate collation is needed.

---

## Dataset Configuration

| Setting | 3-Class Problem | 7-Class Problem |
|---|---|---|
| Classes | NonViolence (0), PreViolence (1), Violence (2) | Pointing (0), CollarGrabbing (1), Pushing (2), Strangling (3), Hitting (4), Headlock (5), KickingAttack (6) |
| `model.blocks[-1].proj` | `nn.Linear(2304, 3)` | `nn.Linear(2304, 7)` |
| CSV label column | `"behavior"` | `"behavior"` |
| Use case | Coarse violence detection | Fine-grained behavior recognition |

> **Switch between 3-class and 7-class** by toggling the `LABEL_MAP` and `nn.Linear(2304, ?)` in the Configuration and Model cells below.

---

## Input Data Format (RGB .npy)

Each video is stored as a `.npy` file with shape **(T, 224, 224, 3)**:
- `T` — number of frames (variable, will be resampled to 32)
- Resolution: 224 × 224
- Channels: 3 (RGB, uint8)

Files are organized under `FEATURE_ROOT/` mirroring the dataset directory tree:
```
FEATURE_ROOT/
  <Class>/
    <Scene>/
      <Angle>/
        [<Behavior>/]     # Violence / PreViolence only
          <video_name>.npy
```

## CSV Format

The CSV must have two columns:
```
video,behavior
Violence/scene01/Top/Hitting/v001,Hitting
NonViolence/scene02/Center/v012,NonViolence
```

## Key Details

| Detail | Value |
|---|---|
| Library | `pytorchvideo` |
| Model | `slowfast_r50(pretrained=False)` |
| Slow frames | 8 (= 32 // 4, stride-4 subsampling) |
| Fast frames | 32 |
| Head | `nn.Linear(2304, num_classes)` |
| Optimizer | AdamW, lr=1e-4 |
| Padding strategy | Repeat last frame (not zeros) |
| Checkpoint | `best_slowfast.pth` |

## Part 1 — Installation

In [ ]:
!pip install pytorchvideo fvcore iopath

## Part 2 — Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from pytorchvideo.models.hub import slowfast_r50

## Part 3 — Configuration

Change `LABEL_MAP` and `nn.Linear(2304, ?)` to switch between **3-class** and **7-class**.

> **CSV label column is `"behavior"`** — make sure your CSV has this column name.

### Frame Sampling Strategy
- If video has ≥ 32 frames → **uniformly sample** 32 frames via `np.linspace`.
- If video has < 32 frames → **repeat the last frame** until reaching 32.

### Pathway Split (no custom collate needed)
```python
fast_pathway = video                  # (B, C, 32, H, W)
slow_pathway = video[:, :, ::4, :, :]  # (B, C,  8, H, W)
```

In [ ]:
# Number of frames per video (Fast pathway)
TARGET_FRAMES = 32

# =============================================================================
# LABEL MAP
# =============================================================================

# ----- 3-Class Setup (coarse violence detection) -----
# Uncomment this block and set nn.Linear(2304, 3) in the Model cell
# LABEL_MAP = {
#     "NonViolence": 0,
#     "PreViolence":  1,
#     "Violence":     2,
# }

# ----- 7-Class Setup (fine-grained behavior recognition) -----
# Uncomment this block and set nn.Linear(2304, 7) in the Model cell
LABEL_MAP = {
    "Pointing":       0,
    "CollarGrabbing": 1,
    "Pushing":        2,
    "Strangling":     3,
    "Hitting":        4,
    "Headlock":       5,
    "KickingAttack":  6,
}

# Root directory containing .npy video files
FEATURE_ROOT = "/content/drive/MyDrive/EarlyViolenceDetection2025/Features/Video_NPY"

print(f"TARGET_FRAMES : {TARGET_FRAMES}")
print(f"NUM_CLASSES   : {len(LABEL_MAP)}")
print(f"LABEL_MAP     : {LABEL_MAP}")

## Part 4 — Dataset

`VideoDataset` loads each `.npy` file and:
1. **Samples** 32 frames uniformly, or **repeats the last frame** if the video is too short.
2. Normalizes pixel values from `[0, 255]` to `[0.0, 1.0]`.
3. Returns a tensor of shape `(C, T, H, W)` = `(3, 32, 224, 224)`.

> **Important:** the CSV label column is `"behavior"`, not `"class"`.

In [ ]:
class VideoDataset(Dataset):
    """
    Loads RGB video clips stored as .npy files.

    Each .npy file has shape (T, 224, 224, 3) — uint8 pixel values.
    The dataset walks FEATURE_ROOT/<Class>/ to locate the matching file.

    CSV required columns:
        video    — relative path of the video clip
        behavior — class label name (must match a key in LABEL_MAP)
    """

    def __init__(self, csv_path, root):
        self.df   = pd.read_csv(csv_path)
        self.root = root

    def find_feature(self, video_rel):
        """Search for the .npy file matching video_rel under FEATURE_ROOT/<Class>/."""
        video_name  = os.path.basename(video_rel)
        npy_name    = os.path.splitext(video_name)[0] + ".npy"
        cls         = video_rel.split("/")[0]
        search_root = os.path.join(self.root, cls)

        for r, _, files in os.walk(search_root):
            if npy_name in files:
                return os.path.join(r, npy_name)
        return None

    def sample_frames(self, video):
        """
        Normalize frame count to TARGET_FRAMES.

        - T >= TARGET_FRAMES : uniform sampling via np.linspace.
        - T <  TARGET_FRAMES : repeat the last frame to pad.

        Args:
            video (np.ndarray): shape (T, 224, 224, 3)
        Returns:
            np.ndarray: shape (TARGET_FRAMES, 224, 224, 3)
        """
        T = video.shape[0]

        if T >= TARGET_FRAMES:
            idx   = np.linspace(0, T - 1, TARGET_FRAMES).astype(int)
            video = video[idx]
        else:
            last_frame = video[-1]
            pad        = np.repeat(last_frame[None, ...], TARGET_FRAMES - T, axis=0)
            video      = np.concatenate([video, pad], axis=0)

        return video

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        video_rel = row["video"]
        label     = LABEL_MAP[row["behavior"]]   # NOTE: column name is "behavior"

        npy_path = self.find_feature(video_rel)
        video    = np.load(npy_path)             # (T, 224, 224, 3)
        video    = self.sample_frames(video)     # (32, 224, 224, 3)
        video    = video.astype(np.float32) / 255.0
        video    = torch.tensor(video).permute(3, 0, 1, 2)  # (C, T, H, W)

        return video, label

## Part 5 — DataLoaders

In [ ]:
train_dataset = VideoDataset("train.csv", FEATURE_ROOT)
val_dataset   = VideoDataset("val.csv",   FEATURE_ROOT)
test_dataset  = VideoDataset("test.csv",  FEATURE_ROOT)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32)
test_loader  = DataLoader(test_dataset,  batch_size=32)

print(f"Train: {len(train_dataset)} samples")
print(f"Val  : {len(val_dataset)} samples")
print(f"Test : {len(test_dataset)} samples")

## Part 6 — Pathway Packing

`pack_pathway_output` splits one video tensor into the two SlowFast inputs **without** a custom collate function:

```python
fast_pathway = video                    # (B, C, 32, H, W)  — all frames
slow_pathway = video[:, :, ::4, :, :]  # (B, C,  8, H, W)  — every 4th frame
```

This is equivalent to SlowFast's alpha=4 temporal stride.

In [ ]:
def pack_pathway_output(video):
    """
    Split a video tensor into [slow, fast] pathway inputs for SlowFast.

    Args:
        video (Tensor): shape (B, C, T, H, W)  where T = TARGET_FRAMES = 32

    Returns:
        list[Tensor]:
            slow_pathway — (B, C, T//4, H, W) = (B, C, 8, H, W)
            fast_pathway — (B, C, T,    H, W) = (B, C, 32, H, W)
    """
    fast_pathway = video
    slow_pathway = video[:, :, ::4, :, :]
    return [slow_pathway, fast_pathway]

## Part 7 — Model

We use `slowfast_r50(pretrained=False)` from `pytorchvideo.models.hub` and replace the
classification head.

> **Head size:** the SlowFast R50 backbone outputs a 2304-dim feature vector.
> Change `nn.Linear(2304, 7)` to `nn.Linear(2304, 3)` for the 3-class setup.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = slowfast_r50(pretrained=True)

# Replace classification head
# For 3-class: nn.Linear(2304, 3)
# For 7-class: nn.Linear(2304, 7)
model.blocks[-1].proj = nn.Linear(2304, 7)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-4,
)

print("Model loaded: SlowFast R50")
print(f"Head: {model.blocks[-1].proj}")

## Part 8 — Training Loop

Early stopping patience = 5 epochs (based on validation loss).
Best model saved as `best_slowfast.pth`.

In [ ]:
EPOCHS  = 50
PATIENCE = 5

best_val_loss    = float("inf")
patience_counter = 0

for epoch in range(EPOCHS):

    # -------------------------------------------------------------- Train
    model.train()

    train_loss   = 0
    y_true_train = []
    y_pred_train = []

    for video, label in tqdm(train_loader):

        video = video.to(device)
        label = label.to(device)

        inputs = pack_pathway_output(video)
        preds  = model(inputs)
        loss   = criterion(preds, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        pred_class = preds.argmax(dim=1)
        y_true_train.extend(label.cpu().numpy())
        y_pred_train.extend(pred_class.cpu().numpy())

    train_loss /= len(train_loader)
    train_acc   = accuracy_score(y_true_train, y_pred_train)

    # ---------------------------------------------------------- Validation
    model.eval()

    val_loss   = 0
    y_true_val = []
    y_pred_val = []

    with torch.no_grad():

        for video, label in val_loader:

            video = video.to(device)
            label = label.to(device)

            inputs = pack_pathway_output(video)
            preds  = model(inputs)
            loss   = criterion(preds, label)

            val_loss += loss.item()

            pred_class = preds.argmax(dim=1)
            y_true_val.extend(label.cpu().numpy())
            y_pred_val.extend(pred_class.cpu().numpy())

    val_loss /= len(val_loader)
    val_acc   = accuracy_score(y_true_val, y_pred_val)

    print(f"\nEpoch {epoch + 1}")
    print("Train loss:", train_loss)
    print("Val loss:  ", val_loss)
    print("Train acc: ", train_acc)
    print("Val acc:   ", val_acc)

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "best_slowfast.pth")
        print("Saved best model")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping")
            break

## Part 9 — Evaluation on Test Set

In [ ]:
model.load_state_dict(torch.load("best_slowfast.pth"))
model.eval()

y_true = []
y_pred = []

with torch.no_grad():

    for video, label in test_loader:

        video = video.to(device)
        label = label.to(device)

        inputs = pack_pathway_output(video)
        preds  = model(inputs)

        pred_class = preds.argmax(dim=1)
        y_true.extend(label.cpu().numpy())
        y_pred.extend(pred_class.cpu().numpy())


acc       = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average="macro")
recall    = recall_score(y_true, y_pred, average="macro")
f1        = f1_score(y_true, y_pred, average="macro")

print("\n===== TEST RESULTS =====")
print("Accuracy :", acc)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-score :", f1)

cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix")
print(cm)

## Part 10 — Confusion Matrix Plot

In [ ]:
plt.figure()
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.colorbar()
plt.show()

## Part 11 — Release GPU (Colab)

In [ ]:
from google.colab import runtime
runtime.unassign()